# EDA: Job Description Structure for RAG Chunking

**Dataset:** LF Jobs (1000 listings, 9 columns)  
**Goal:** Decide the best way to clean and chunk **Job Description** HTML so a RAG pipeline returns relevant *job listings*.

This notebook records the exploratory findings that led to the hybrid structure-aware chunker.


## 1. Load & basic overview

In [ ]:
import re
from collections import Counter
from html import unescape

import pandas as pd
from bs4 import BeautifulSoup

DATA_PATH = "/home/arun-joshi/Projects/Leapfrog/RAG/data/LF_Jobs.xlsx"

df = pd.read_excel(DATA_PATH)
df = df.fillna("")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
print("Job Category distribution:")
print(df["Job Category"].value_counts())
print()
print("Job Level distribution:")
print(df["Job Level"].value_counts())
print()
print("Unique companies:", df["Company Name"].nunique())


Shape: (1000, 9)
Columns: ['ID', 'Job Category', 'Job Title', 'Company Name', 'Publication Date', 'Job Location', 'Job Level', 'Tags', 'Job Description']

Job Category distribution:
Job Category
Data and Analytics           166
Software Engineering         166
Design and UX                166
Sales                        166
Project Management           166
Advertising and Marketing    166
General                        4
Name: count, dtype: int64

Job Level distribution:
Job Level
Senior Level    610
Mid Level       349
Internship       24
Entry Level      17
Name: count, dtype: int64

Unique companies: 145


## 2. Description length distribution

Raw HTML length vs cleaned text length (tags stripped).  
Most descriptions sit in the 4k–7k character range after cleaning → multiple chunks per job are expected.


In [2]:
def clean_len(html: str) -> int:
    if not html:
        return 0
    soup = BeautifulSoup(str(html), "html.parser")
    text = re.sub(r"\s+", " ", soup.get_text(" ")).strip()
    return len(text)

raw_lens = df["Job Description"].astype(str).str.len()
clean_lens = df["Job Description"].apply(clean_len)

summary = pd.DataFrame({
    "raw_html": raw_lens.describe(),
    "cleaned_text": clean_lens.describe(),
})
print(summary.round(0))
print()
print("Cleaned length percentiles:")
for p in [5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p:>2}%: {clean_lens.quantile(p/100):.0f} chars")


       raw_html  cleaned_text
count    1000.0        1000.0
mean     5755.0        5262.0
std      2005.0        1846.0
min       727.0         589.0
25%      4417.0        4049.0
50%      5616.0        5140.0
75%      7052.0        6445.0
max     17051.0       15713.0

Cleaned length percentiles:
   5%: 2428 chars
  10%: 2961 chars
  25%: 4049 chars
  50%: 5140 chars
  75%: 6445 chars
  90%: 7555 chars
  95%: 8154 chars
  99%: 10671 chars


## 3. Heading / markup style inventory

Real semantic headings (`<h1>`–`<h6>`) are almost never used.  
The dominant pattern is informal bolding: `<br><b>…</b>`, `<br><br><b>…</b>`, or `<p><strong>…</strong>`.


In [3]:
def style_bucket(html: str) -> str:
    t = str(html)
    has_h = bool(re.search(r"<h[1-6]", t, re.I))
    has_br_b = bool(
        re.search(
            r"<br\s*/?\s*>\s*(?:<br\s*/?\s*>\s*)*<(?:b|strong)>",
            t,
            re.I,
        )
    )
    has_p_b = bool(re.search(r"<p[^>]*>\s*<(?:b|strong)>", t, re.I))
    has_b = "<b>" in t.lower() or "<strong>" in t.lower()
    if has_h:
        return "uses_<h*>"
    if has_br_b:
        return "uses_<br>+<b/strong>"
    if has_p_b:
        return "uses_<p>+<b/strong>"
    if has_b:
        return "uses_bold_elsewhere"
    return "no_bold_or_heading"

styles = df["Job Description"].apply(style_bucket)
print(styles.value_counts())
print()
print((styles.value_counts(normalize=True) * 100).round(1).astype(str) + "%")


Job Description
uses_<br>+<b/strong>    853
no_bold_or_heading       42
uses_bold_elsewhere      39
uses_<p>+<b/strong>      37
uses_<h*>                29
Name: count, dtype: int64

Job Description
uses_<br>+<b/strong>    85.3%
no_bold_or_heading       4.2%
uses_bold_elsewhere      3.9%
uses_<p>+<b/strong>      3.7%
uses_<h*>                2.9%
Name: proportion, dtype: str


## 4. Pseudo-section extraction quality

Treat short bold spans after line breaks as candidate section headers.  
Count how many such sections appear per job and how long their content is.


In [4]:
HEADING_RE = re.compile(
    r"(?:<br\s*/?\s*>\s*)*(?:<p[^>]*>\s*)?<(b|strong|h[1-6])(?:\s+[^>]*)?>(.*?)</\1>",
    re.I | re.DOTALL,
)

def extract_raw_headers(html: str):
    out = []
    for m in HEADING_RE.finditer(str(html)):
        text = re.sub(r"<[^>]+>", "", m.group(2))
        text = re.sub(r"\s+", " ", text).strip()
        if 2 <= len(text) <= 100:
            out.append(text)
    return out

header_counts = df["Job Description"].apply(lambda h: len(extract_raw_headers(h)))
print("Pseudo-headers per job:")
print(header_counts.describe().round(2))
print()
print("Jobs with 0 headers:", (header_counts == 0).sum())
print("Jobs with 1–3:     ", ((header_counts >= 1) & (header_counts <= 3)).sum())
print("Jobs with 4–8:     ", ((header_counts >= 4) & (header_counts <= 8)).sum())
print("Jobs with 9+:      ", (header_counts >= 9).sum())


Pseudo-headers per job:
count    1000.00
mean        7.62
std         5.29
min         0.00
25%         4.00
50%         6.00
75%        10.00
max        46.00
Name: Job Description, dtype: float64

Jobs with 0 headers: 45
Jobs with 1–3:      116
Jobs with 4–8:      490
Jobs with 9+:       349


In [5]:
all_headers = []
for html in df["Job Description"]:
    all_headers.extend(extract_raw_headers(html))

normalized = [
    re.sub(r"\s+", " ", h).strip(" :.-•").lower() for h in all_headers
]
print("Top 25 most common bold/header texts:")
for h, c in Counter(normalized).most_common(25):
    print(f"  {c:4d}  |  {h}")


Top 25 most common bold/header texts:
   203  |  preferred qualifications
   193  |  responsibilities
   163  |  qualifications
   144  |  minimum qualifications
   135  |  job description
   112  |  what you'll do
    98  |  about us
    87  |  position summary
    86  |  pay range
    76  |  requirements
    75  |  about the team
    69  |  required qualifications
    69  |  job responsibilities
    68  |  who we are
    66  |  about meta
    65  |  education
    58  |  401(k) plan
    57  |  great benefits for great people
    57  |  affordable medical plan options,
    57  |  employee stock purchase plan
    57  |  no-cost programs for all colleagues
    57  |  benefit solutions that address the different needs and preferences of our colleagues
    57  |  we offer
    56  |  #li-dni
    55  |  primary location


## 5. List density

Most descriptions contain bullet lists (responsibilities, requirements, skills).  
Chunking must try to keep list items together.


In [6]:
def list_stats(html: str):
    soup = BeautifulSoup(str(html), "html.parser")
    return len(soup.find_all(["ul", "ol"])), len(soup.find_all("li"))

stats = df["Job Description"].apply(list_stats)
n_uls = [u for u, _ in stats]
n_lis = [l for _, l in stats]
print(f"Jobs containing at least one list: {sum(u > 0 for u in n_uls)} / {len(df)}")
print(f"Avg <ul>/<ol> per job: {sum(n_uls)/len(df):.2f}")
print(f"Avg <li> per job:      {sum(n_lis)/len(df):.2f}")
print(f"Max <li> in one job:   {max(n_lis)}")


Jobs containing at least one list: 824 / 1000
Avg <ul>/<ol> per job: 3.09
Avg <li> per job:      17.35
Max <li> in one job:   79


## 6. Boilerplate / legal density

EEO, disability, veteran, NMLS, Dodd-Frank language appears frequently.  
These blocks add little retrieval value and should be filtered.


In [7]:
BOILER_KW = [
    "equal opportunity", "equal employment opportunity", "affirmative action",
    "all qualified applicants", "without regard to", "regard to race",
    "sexual orientation", "gender identity", "protected veteran",
    "disability status", "reasonable accommodat", "eeo",
    "dodd frank", "truth in lending", "nmls", "criminal conviction",
]

def boiler_hits(html: str) -> int:
    t = str(html).lower()
    return sum(1 for kw in BOILER_KW if kw in t)

hits = df["Job Description"].apply(boiler_hits)
print("Jobs with ≥1 boilerplate keyword:", (hits >= 1).sum())
print("Jobs with ≥3 boilerplate keywords:", (hits >= 3).sum())
print("Jobs with ≥5 boilerplate keywords:", (hits >= 5).sum())
print()
print(hits.describe().round(2))


Jobs with ≥1 boilerplate keyword: 603
Jobs with ≥3 boilerplate keywords: 489
Jobs with ≥5 boilerplate keywords: 327

count    1000.00
mean        3.01
std         3.12
min         0.00
25%         0.00
50%         2.00
75%         6.00
max         9.00
Name: Job Description, dtype: float64


## 7. Noise headers that look like sections but are not

Examples that should be rejected as section titles: tracking tags, emails, relocation flags, requisition IDs.


In [8]:
NOISE_RE = re.compile(
    r"(?i)^(#li[-_].*|reqid.*|go\.[a-z0-9./-]+|"
    r"[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}|"
    r"relocation (assistance|package).*|nearest major market.*|"
    r"work shift.*|additional locations?|fair chance.*|requisition code.*)$"
)

noise, clean = [], []
for h in all_headers:
    key = h.strip()
    if NOISE_RE.match(key) or re.fullmatch(r"[\d\s\-_/]+", key):
        noise.append(key)
    else:
        clean.append(key)

print(f"Noise-like headers found: {len(noise)}")
print("Sample noise headers:")
for h in list(dict.fromkeys(noise))[:15]:
    print(f"  - {h}")
print()
print(f"Remaining candidate headers: {len(clean)}")


Noise-like headers found: 192
Sample noise headers:
  - ombuds.person@wipro.com
  - #LI-DNI
  - Relocation Assistance Provided:
  - #LI-AL15
  - go.atlassian.com/perksandbenefits
  - go.atlassian.com/crh
  - ReqID
  - Relocation Package
  - Fair Chance Hiring and Ban-the-Box Notices | Deloitte US Careers
  - #LI-HK1
  - #LI-TG3
  - Nearest Major Market:
  - #LI-SG6
  - Additional Locations
  - Work Shift

Remaining candidate headers: 7426


## 8. Section content length after extraction

Once noise headers are removed, how big are the actual content blocks?


In [9]:
def section_content_lengths(html: str):
    html = str(html)
    matches = list(HEADING_RE.finditer(html))
    lengths = []
    if not matches:
        return [clean_len(html)]
    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(html)
        lengths.append(clean_len(html[start:end]))
    return lengths

all_sec_lens = []
for html in df["Job Description"]:
    all_sec_lens.extend(section_content_lengths(html))

s = pd.Series(all_sec_lens)
print("Section content length stats:")
print(s.describe().round(0))
print()
print("Percentiles:")
for p in [10, 25, 50, 75, 90]:
    print(f"  {p}%: {s.quantile(p/100):.0f}")


Section content length stats:
count    8148.0
mean      563.0
std       805.0
min         0.0
25%        38.0
50%       314.0
75%       778.0
max      9887.0
dtype: float64

Percentiles:
  10%: 0
  25%: 38
  50%: 314
  75%: 778
  90%: 1358


## 9. Implications for chunking (summary)

| Finding | Decision |
|---------|----------|
| Real `<h*>` rare; `<br><b>` / <strong> dominate | Treat bold-after-break as pseudo-headings |
| ~4 headers/job median; some jobs have 0 | Structure-first, size/semantic fallback |
| Median section ~500 chars; some >2k | Keep small sections whole; size-split large ones |
| Heavy lists | Prefer not to split mid-list |
| Frequent EEO/legal tails + noise tags | Filter boilerplate & noise headers |
| Goal = return *job listings* | Prefix every chunk with Job ID / Title / Company / Level / Location / Section |
| ~5–6 chunks/job → ~5–6k total | Normal; use top_k 20–40 + dedupe by job_id |


## 10. Quick preview of a few cleaned sections

Sanity-check what the extractor sees on real rows.


In [10]:
def preview_sections(idx: int, max_sections: int = 6):
    row = df.iloc[idx]
    html = str(row["Job Description"])
    headers = extract_raw_headers(html)
    print(f"=== [{row['ID']}] {row['Job Title'][:60]} @ {row['Company Name']} ===")
    print(f"Level: {row['Job Level']} | Location: {row['Job Location']}")
    print(f"Detected headers ({len(headers)}):")
    for h in headers[:max_sections]:
        print(f"  • {h}")
    if len(headers) > max_sections:
        print(f"  … +{len(headers) - max_sections} more")
    print()

for i in [0, 3, 7, 50, 200]:
    preview_sections(i)


=== [LF0001] DIR, Equities Quant @ Merrill ===
Level: Mid Level | Location: New York, NY
Detected headers (8):
  • Job Description:
  • Job Description:
  • Responsibilities:
  • Skills:
  • Minimum Education Requirement:
  • Shift:
  … +2 more

=== [LF0004] Retail Sales Supervisor- Melrose Studio @ Soho House ===
Level: Senior Level | Location: West Hollywood, CA
Detected headers (12):
  • The Role…
  • Main Duties…
  • Requirements...
  • Physical Requirements
  • Why work with us...
  • Health Care + 401K:&nbsp;
  … +6 more

=== [LF0008] Senior Data Engineer @ Fisher Investments ===
Level: Senior Level | Location: Camas, WA
Detected headers (6):
  • Overview
  • The Opportunity:
  • The Day-to-Day:
  • Your Qualifications:
  • Compensation:
  • Why Fisher Investments:

=== [LF0051] Quantitative Researcher, Growth @ Meta ===
Level: Mid Level | Location: Tel Aviv, Israel
Detected headers (4):
  • Quantitative Researcher, Growth Responsibilities:
  • Minimum Qualifications:
  • Preferr